# Analiza jakości danych surowych (orders_train_raw.csv)

Ten notatnik wykonuje wstępną inspekcję jakości zbioru treningowego `data/raw/orders_train_raw.csv` bez modyfikacji pliku źródłowego.

### Zakres kontroli:
- Liczba wierszy i unikalnych dat
- Wykrywanie powtórzonych wierszy (duplikatów)
- Wykrywanie brakujących wartości (pustych komórek)
- Wykrywanie ujemnych wartości w kolumnie `orders`
- Wykrywanie tekstowych wartości w kolumnie `promo`

In [1]:
import pandas as pd
from pathlib import Path

# Sprawdzamy ścieżkę do pliku źródłowego
path = Path('../data/raw/orders_train_raw.csv') if Path('../data/raw/orders_train_raw.csv').exists() else Path('data/raw/orders_train_raw.csv')
df = pd.read_csv(path, encoding='utf-8')

print('Wczytano plik pomyślnie.')
df.info()

Wczytano plik pomyślnie.
<class 'pandas.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  255 non-null    str    
 1   promo                 255 non-null    str    
 2   planned_ad_spend_pln  251 non-null    float64
 3   orders                253 non-null    float64
 4   visits                255 non-null    int64  
 5   revenue_pln           255 non-null    float64
dtypes: float64(3), int64(1), str(2)
memory usage: 12.1 KB


In [2]:
# 1. Pierwsze 10 wierszy zbioru danych
df.head(10)

,date,promo,planned_ad_spend_pln,orders,visits,revenue_pln
0,2024-01-01,0,852.81,88.0,841,9090.32
1,2024-01-02,1,413.23,84.0,737,9404.07
2,2024-01-03,0,961.22,77.0,987,6579.19
3,2024-01-04,0,697.53,84.0,1046,7581.46
4,2024-01-05,0,541.13,76.0,1069,7947.15
5,2024-01-06,0,229.25,100.0,1022,11890.46
6,2024-01-07,1,513.96,108.0,968,11229.29
7,2024-01-08,0,998.01,88.0,1171,10047.78
8,2024-01-09,0,393.78,62.0,794,7113.78
9,2024-01-10,0,404.74,66.0,742,6928.37


In [3]:
# 2. Podsumowanie wymiarów i unikalności dat
n_wierszy = len(df)
n_dat = df['date'].nunique()
n_duplikatow_wierszy = df.duplicated().sum()
n_duplikatow_dat = df['date'].duplicated().sum()

podsumowanie_wymiarow = pd.DataFrame({
    'Metryka': ['Liczba wierszy', 'Liczba unikalnych dat', 'Powtórzone wiersze (pełne)', 'Powtórzone daty'],
    'Wartość': [n_wierszy, n_dat, n_duplikatow_wierszy, n_duplikatow_dat]
})
podsumowanie_wymiarow

,Metryka,Wartość
0,Liczba wierszy,255
1,Liczba unikalnych dat,252
2,Powtórzone wiersze (pełne),3
3,Powtórzone daty,3


In [4]:
# 3. Puste komórki (braki danych) w poszczególnych kolumnach
braki_df = pd.DataFrame({
    'Kolumna': df.columns,
    'Liczba braków (NaN)': df.isna().sum().values,
    'Procent braków (%)': (df.isna().sum().values / len(df) * 100).round(2)
})
braki_df

,Kolumna,Liczba braków (NaN),Procent braków (%)
0,date,0,0.00
1,promo,0,0.00
2,planned_ad_spend_pln,4,1.57
3,orders,2,0.78
4,visits,0,0.00
5,revenue_pln,0,0.00


In [5]:
# 4. Szczegóły braków w kolumnie planned_ad_spend_pln oraz orders
print('Braki w planned_ad_spend_pln:')
display(df[df['planned_ad_spend_pln'].isna()])

print('Braki w orders:')
display(df[df['orders'].isna()])

Braki w planned_ad_spend_pln:


,date,promo,planned_ad_spend_pln,orders,visits,revenue_pln
17,2024-01-18,0,NaN,87.0,761,8664.65
89,2024-03-30,0,NaN,115.0,1501,10509.63
170,2024-06-19,0,NaN,55.0,480,4407.89
248,2024-09-05,0,NaN,71.0,949,7653.26


Braki w orders:


,date,promo,planned_ad_spend_pln,orders,visits,revenue_pln
63,2024-03-04,1,732.12,NaN,1069,8335.70
192,2024-07-11,0,763.38,NaN,1334,8280.79


In [6]:
# 5. Weryfikacja poprawności kolumny orders (ujemne wartości)
# Ujemne orders traktujemy jako ewidentny błąd danych
ujemne_orders = df[pd.to_numeric(df['orders'], errors='coerce') < 0]
print(f'Liczba ujemnych wartości orders: {len(ujemne_orders)}')
ujemne_orders

Liczba ujemnych wartości orders: 2


,date,promo,planned_ad_spend_pln,orders,visits,revenue_pln
120,2024-04-30,0,440.53,-5.0,746,6561.21
225,2024-08-13,0,767.83,-5.0,940,8093.25


In [7]:
# 6. Weryfikacja kolumny promo (sprawdzenie wartości tekstowych)
print('Rozkład wartości w promo:')
print(df['promo'].value_counts(dropna=False))

# Odnalezienie wierszy z tekstem (np. ' yes ')
tekstowe_promo = df[~df['promo'].astype(str).str.strip().isin(['0', '1'])]
print(f'\nLiczba wierszy z niespójnym promo: {len(tekstowe_promo)}')
tekstowe_promo

Rozkład wartości w promo:
promo
0        203
1         49
 yes       3
Name: count, dtype: int64

Liczba wierszy z niespójnym promo: 3


,date,promo,planned_ad_spend_pln,orders,visits,revenue_pln
40,2024-02-10,yes,205.78,101.0,1440,9929.48
123,2024-05-03,yes,271.90,112.0,966,11481.11
201,2024-07-20,yes,445.64,106.0,1405,11356.94


In [8]:
# 7. Identyfikacja powtórzonych wierszy
duplikaty = df[df.duplicated(keep=False)].sort_values('date')
print(f'Liczba powtórzonych wystąpień: {len(duplikaty)}')
duplikaty

Liczba powtórzonych wystąpień: 6


,date,promo,planned_ad_spend_pln,orders,visits,revenue_pln
30,2024-01-31,0,896.19,85.0,919,9355.78
252,2024-01-31,0,896.19,85.0,919,9355.78
150,2024-05-30,0,246.24,52.0,732,6025.31
253,2024-05-30,0,246.24,52.0,732,6025.31
230,2024-08-18,0,928.43,102.0,865,9145.95
254,2024-08-18,0,928.43,102.0,865,9145.95


## 8. Statystyki opisowe dla poprawnych wartości orders

Obliczenie miar tendencji centralnej i rozproszenia (średnia, mediana, minimum, maksimum, kwartyle Q1 i Q3) dla poprawnych zamówień wraz z liczbą uwzględnionych dni.

In [9]:
# 8. Obliczenie statystyk dla poprawnych orders ze zbioru oczyszczonego
path_cleaned = Path('../data/processed/orders_train.csv') if Path('../data/processed/orders_train.csv').exists() else Path('data/processed/orders_train.csv')
df_cleaned = pd.read_csv(path_cleaned)

# Filtrujemy tylko poprawne (niepuste i nieujemne) wartości orders
poprawne_orders = df_cleaned['orders'].dropna()

statystyki_orders = pd.DataFrame({
    'Liczba uwzględnionych dni': [int(poprawne_orders.count())],
    'Średnia': [round(poprawne_orders.mean(), 2)],
    'Mediana': [poprawne_orders.median()],
    'Minimum': [poprawne_orders.min()],
    'Maksimum': [poprawne_orders.max()],
    'Kwartyl 1 (25%)': [poprawne_orders.quantile(0.25)],
    'Kwartyl 3 (75%)': [poprawne_orders.quantile(0.75)]
})

statystyki_orders


,Liczba uwzględnionych dni,Średnia,Mediana,Minimum,Maksimum,Kwartyl 1 (25%),Kwartyl 3 (75%)
0,248,84.41,85.0,38.0,138.0,69.0,98.25


## 9. Ręczny rachunek dla próby 5 dni (Weryfikacja kalkulatorem)

Do ręcznej weryfikacji wybrano pierwsze 5 dni z poprawną liczbą zamówień (`orders`):

| Lp. | Data (`date`) | Liczba zamówień (`orders`) |
|:---:|:---:|:---:|
| 1 | 2024-01-01 | 88.0 |
| 2 | 2024-01-02 | 84.0 |
| 3 | 2024-01-03 | 77.0 |
| 4 | 2024-01-04 | 84.0 |
| 5 | 2024-01-05 | 76.0 |

### Obliczenia krok po kroku (kalkulator):
* **Suma zamówień:** $88.0 + 84.0 + 77.0 + 84.0 + 76.0 = 409.0$
* **Liczba dni:** $N = 5$
* **Średnia arytmetyczna:** $\bar{x} = \frac{409.0}{5} = \mathbf{81.8}$
* **Mediana:**
  * Wartości posortowane niemalejąco: $[76.0, 77.0, \mathbf{84.0}, 84.0, 88.0]$
  * Środkowy element (pozycja 3 przy $N=5$): $\mathbf{84.0}$

In [10]:
# 9. Sprawdzenie dokładności ręcznego rachunku w pandas dla 5 wybranych dni
proba_5_dni = df_cleaned.head(5)[['date', 'orders']]

srednia_kalkulator = 81.8
mediana_kalkulator = 84.0

srednia_pandas = proba_5_dni['orders'].mean()
mediana_pandas = proba_5_dni['orders'].median()

print("Wybrane 5 rekordów:")
print(proba_5_dni.to_string(index=False))
print(f"\nŚrednia kalkulator: {srednia_kalkulator} | Średnia pandas: {srednia_pandas}")
print(f"Mediana kalkulator: {mediana_kalkulator} | Mediana pandas: {mediana_pandas}")

assert srednia_kalkulator == srednia_pandas, 'Niezgodność średniej!'
assert mediana_kalkulator == mediana_pandas, 'Niezgodność mediany!'
print('\n✓ Wyniki obliczeń ręcznych są w 100% zgodne z pandas!')


Wybrane 5 rekordów:
      date  orders
2024-01-01    88.0
2024-01-02    84.0
2024-01-03    77.0
2024-01-04    84.0
2024-01-05    76.0

Średnia kalkulator: 81.8 | Średnia pandas: 81.8
Mediana kalkulator: 84.0 | Mediana pandas: 84.0

✓ Wyniki obliczeń ręcznych są w 100% zgodne z pandas!
